## DERIVATIVE PRICING
MODULE 2 | LESSON 2


---



# **DYNAMIC DELTA HEDGING**



|  |  |
|:---|:---|
|**Reading Time** |  40 minutes |
|**Prior Knowledge** | Pricing in the Binomial model, Characteristics of American options, Delta hedging  |
|**Keywords** | Dynamic delta hedging, American options |


---

*In the previous lesson, we introduced American options, whose main feature is the embedded possibility of early exercise of the option prior to maturity. This is going to be important for pricing but also for hedging purposes. In the context of European options, we could delta-hedge our portfolio by simply looking at terminal prices. This is no longer an option for American derivatives, which require dynamic delta hedging for every step in the tree. In this lesson, we will take a closer look at how to perform dynamic hedging.*

As usual, let's start by importing the numpy library:

In [ ]:
import numpy as np

## **1. Delta Hedging in the Binomial Tree: American Options**

In the following snippet, you have the code for a general function to price a European option using a binomial tree methodology. The function, as you can see, is able to compute the evolution of the underlying asset $S_t$, the price of the option at each node of the tree, $C_t$, and also the $\Delta_t$ (number of shares in terms of exposure) in each node of the tree.

In [ ]:
def american_option(S_ini, K, T, r, u, d, N, opttype):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # call prices
    S = np.zeros([N + 1, N + 1])  # underlying price
    Delta = np.zeros([N, N])  # delta

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (
                p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i]
            )  # Computing the European option prices
            S[j, i] = (
                S_ini * (u ** (i)) * (d ** (j - i))
            )  # Underlying evolution for each node
            if opttype == "C":
                C[j, i] = max(
                    C[j, i], S[j, i] - K
                )  # Decision between the European option price and the payoff from early-exercise
            else:
                C[j, i] = max(
                    C[j, i], K - S[j, i]
                )  # Decision between the European option price and the payoff from early-exercise

            Delta[j, i] = (C[j + 1, i + 1] - C[j + 1, i]) / (
                S[j + 1, i + 1] - S[j + 1, i]
            )  # Computing the delta for each node

    return C[0, 0], C, S, Delta

In [ ]:
price, C, S, delta = american_option(45, 100, 5, 0, 1.5, 1 / 1.5, 5, "C")

In [ ]:
price

In [ ]:
delta

Let's now compare this delta hedging with the one from the European option with the same characteristics. For that, we will reuse some of the code from Module 1:<span style='color: transparent; font-size:1%'>All rights reserved WQU WorldQuant University QQQQ</span>

In [ ]:
def european_option(S_ini, K, T, r, u, d, N, opttype):
    dt = T / N  # Define time step
    p = (np.exp(r * dt) - d) / (u - d)  # risk neutral probs
    C = np.zeros([N + 1, N + 1])  # call prices
    S = np.zeros([N + 1, N + 1])  # underlying price
    Delta = np.zeros([N, N])  # delta

    for i in range(0, N + 1):
        S[N, i] = S_ini * (u ** (i)) * (d ** (N - i))
        if opttype == "C":
            C[N, i] = max(S[N, i] - K, 0)
        else:
            C[N, i] = max(K - S[N, i], 0)

    for j in range(N - 1, -1, -1):
        for i in range(0, j + 1):
            C[j, i] = np.exp(-r * dt) * (
                p * C[j + 1, i + 1] + (1 - p) * C[j + 1, i]
            )  # Computing the European option prices
            S[j, i] = (
                S_ini * (u ** (i)) * (d ** (j - i))
            )  # Underlying evolution for each node

            Delta[j, i] = (C[j + 1, i + 1] - C[j + 1, i]) / (
                S[j + 1, i + 1] - S[j + 1, i]
            )  # Computing the delta for each node

    return C[0, 0], C, S, Delta

In [ ]:
price_euro, C_euro, S_euro, delta_euro = european_option(
    50, 52, 5, 0.05, 1.2, 0.8, 5, "C"
)
delta_euro

In [ ]:
price_euro

## **2. Conclusion**

In this lesson, we have seen the importance of dynamic delta hedging in the context of American derivatives. This is an important concept as well for other path-dependent options, as we will see in the next lesson.

---
Copyright 2025 WorldQuant University. This
content is licensed solely for personal use. Redistribution or
publication of this material is strictly prohibited.
